# Transfer Learning-Based Classification of Alzheimer’s and Parkinson’s Diseases
### Using ResNet50 and EfficientNet-B0 with structural MRI

## 1. Environment & Data Setup

In [ ]:
import os
import shutil
import pathlib
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print("TensorFlow version:", tf.__version__)

### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Extract Datasets
Ensure your ZIP files are in your Google Drive.

In [ ]:
# Update these paths to your actual Google Drive locations
# Based on your feedback, your dataset is in 'Colab_Notebooks_Data'
ALZ_ZIP = '/content/drive/MyDrive/Colab_Notebooks_Data/alzheimer_data.zip'
PD_ZIP = '/content/drive/MyDrive/Colab_Notebooks_Data/parkinsons_data.zip'

!unzip -q "$ALZ_ZIP" -d /content/alzheimer_raw
!unzip -q "$PD_ZIP" -d /content/parkinsons_raw

## 2. Data Processing Pipeline

In [ ]:
def setup_alzheimer_binary(raw_dir, target_dir):
    if os.path.exists(target_dir): shutil.rmtree(target_dir)
    os.makedirs(os.path.join(target_dir, "Non-Demented"), exist_ok=True)
    os.makedirs(os.path.join(target_dir, "Demented"), exist_ok=True)
    
    mappings = {"Non Demented": "Non-Demented", "Very mild Dementia": "Demented", 
                "Mild Dementia": "Demented", "Moderate Dementia": "Demented"}
    
    for src_cat, tgt_cat in mappings.items():
        path = os.path.join(raw_dir, src_cat)
        if not os.path.exists(path): continue
        for img in os.listdir(path):
            shutil.copy(os.path.join(path, img), os.path.join(target_dir, tgt_cat, f"{src_cat}_{img}"))

def setup_parkinsons_binary(raw_dir, target_dir):
    if os.path.exists(target_dir): shutil.rmtree(target_dir)
    os.makedirs(os.path.join(target_dir, "Normal"), exist_ok=True)
    os.makedirs(os.path.join(target_dir, "Parkinson"), exist_ok=True)
    
    # Note: adjust paths based on actual ZIP structure
    pd_data_path = os.path.join(raw_dir, 'parkinsons_dataset')
    for cat in ['normal', 'parkinson']:
        tgt = 'Normal' if cat == 'normal' else 'Parkinson'
        path = os.path.join(pd_data_path, cat)
        for img in os.listdir(path):
            shutil.copy(os.path.join(path, img), os.path.join(target_dir, tgt, img))

# Setup
setup_alzheimer_binary('/content/alzheimer_raw', '/content/processed/alzheimer')
setup_parkinsons_binary('/content/parkinsons_raw', '/content/processed/parkinsons')

In [ ]:
def create_generators(data_dir, img_size=(224, 224), batch_size=32):
    filepaths, labels = [], []
    for category in os.listdir(data_dir):
        path = os.path.join(data_dir, category)
        for img in os.listdir(path):
            filepaths.append(os.path.join(path, img))
            labels.append(category)
    
    df = pd.DataFrame({'filepath': filepaths, 'label': labels})
    train_df, test_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
    valid_df, test_df = train_test_split(test_df, test_size=0.5, stratify=test_df['label'], random_state=42)
    
    train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, horizontal_flip=True, zoom_range=0.1)
    val_test_datagen = ImageDataGenerator(rescale=1./255)
    
    train_gen = train_datagen.flow_from_dataframe(train_df, x_col='filepath', y_col='label', target_size=img_size, batch_size=batch_size, class_mode='binary')
    valid_gen = val_test_datagen.flow_from_dataframe(valid_df, x_col='filepath', y_col='label', target_size=img_size, batch_size=batch_size, class_mode='binary', shuffle=False)
    test_gen = val_test_datagen.flow_from_dataframe(test_df, x_col='filepath', y_col='label', target_size=img_size, batch_size=batch_size, class_mode='binary', shuffle=False)
    
    weights = class_weight.compute_class_weight('balanced', classes=np.unique(train_gen.classes), y=train_gen.classes)
    return train_gen, valid_gen, test_gen, dict(enumerate(weights))

alz_train, alz_val, alz_test, alz_weights = create_generators('/content/processed/alzheimer')
pd_train, pd_val, pd_test, pd_weights = create_generators('/content/processed/parkinsons')

## 3. Model Architectures

In [ ]:
def build_model(base_type='resnet50', input_shape=(224, 224, 3)):
    if base_type == 'resnet50':
        base = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    else:
        base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape)
    
    base.trainable = False
    model = models.Sequential([
        base, 
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

## 4. Training Infrastructure

In [ ]:
def train(model, train_gen, val_gen, weights, name):
    callbacks = [
        EarlyStopping(patience=5, restore_best_weights=True),
        ModelCheckpoint(f'best_{name}.h5', save_best_only=True),
        ReduceLROnPlateau(factor=0.2, patience=3)
    ]
    return model.fit(train_gen, validation_data=val_gen, epochs=20, class_weight=weights, callbacks=callbacks)

## 5. Evaluation

In [ ]:
def evaluate(model, test_gen, name):
    preds = model.predict(test_gen)
    y_pred = (preds > 0.5).astype(int)
    y_true = test_gen.classes
    
    print(f"--- {name} Results ---")
    print(classification_report(y_true, y_pred))
    
    cm = confusion_matrix(y_true, y_pred)
    plt.imshow(cm, cmap='Blues')
    plt.title(f'CM - {name}')
    plt.show()

## 6. Explainability (Grad-CAM)

In [ ]:
def grad_cam(model, img_array, layer_name):
    grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

## 7. Execution

In [ ]:
# Example for Alzheimer's
model_alz = build_model('resnet50')
history_alz = train(model_alz, alz_train, alz_val, alz_weights, 'alz_resnet')
evaluate(model_alz, alz_test, 'Alzheimer ResNet50')